
# Transform Order Data 

1. Pre-processing the JSON string fix the Data Quality Issue
2. Transform JSON string to JSON Object
3. Write transformed data to silver schema


## 1. Pre-processing the JSON string fix the Data Quality Issue

In [0]:
SELECT *, 
       regexp_replace(value, '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date":"\$1"')
FROM gizmobox.bronze.v_orders


## 2. Transform JSON String to JSON Object

1. To convert to JSON object we need to mention the schema, FUNCTION schema_of_json
2. Next, FUNCTION from_json converts the string to JSON object

In [0]:
SELECT schema_of_json(value)
FROM gizmobox.bronze.v_orders LIMIT 1;

-- So, we got the schema of the json string

In [0]:
SELECT from_json(
  value, 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details:
        STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, 
        quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, 
        payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>')
    as json_value
FROM gizmobox.bronze.v_orders
LIMIT 1; 


## 3. Write transformed data to the silver schema

In [0]:
CREATE TABLE IF NOT EXISTS gizmobox.silver.orders_json
AS
SELECT from_json(
  value, 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details:
        STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, 
        quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, 
        payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>')
    as json_value
FROM gizmobox.bronze.v_orders;

In [0]:
SELECT * FROM gizmobox.silver.orders_json